\# New Release Crawler (v0.7)

This notebook monitors financial regulation sites for new releases.

**Sites Monitored:**
- FSB (Financial Stability Board)
- BCBS (Basel Committee) - *Uses Selenium*
- NGFS (Network for Greening the Financial System)
- PCAF (Partnership for Carbon Accounting Financials) Global & Korea
- PCAF Korea (Kosif) - *Uses Selenium*
- FSC (Financial Services Commission - Korea)
- FSS (Financial Supervisory Service - Korea)

In [ ]:
# Install dependencies
!pip install schedule beautifulsoup4 requests selenium webdriver-manager

# Install Google Chrome Stable
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable

-----------------------------------------------------------

In [ ]:
import requests
import json
import os
import time
import datetime
import schedule
import sys
import re
from bs4 import BeautifulSoup
from abc import ABC, abstractmethod
from typing import List, Dict, Any, Set
from google.colab import drive
import shutil

# Selenium setup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def get_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    # Use a more recent User-Agent and try to hide automation
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
    options.add_argument('--disable-blink-features=AutomationControlled')

    # Standard robust install for Colab + Linux
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
        return driver
    except Exception as e:
        print(f"Driver Init Error: {e}")
        return None

In [ ]:
drive.mount('/content/drive')

# Define recipient emails as a list globally
recipient_emails = [
     "noheulhong@kr.kpmg.com",
     "taeyikim@kr.kpmg.com",
      "yurimkoo@kr.kpmg.com",
      "inhyelee@kr.kpmg.com",
      "taemusun@kr.kpmg.com",
    "woochuljung@kr.kpmg.com",

    "minhocho@kr.kpmg.com"

]

# 저장했던 파일 경로 (본인이 설정한 경로와 동일해야 함)
file_path = '/content/drive/MyDrive/history.json'

# Initial load of history data (will be handled by JsonStorage, but kept for context if needed elsewhere)
# with open(file_path, 'r', encoding='utf-8') as f:
#     loaded_data = json.load(f)

# 불러온 데이터 확인 (Optional, for debugging initial load)
# print(loaded_data)

In [ ]:
import requests
import json
import os
import time
import datetime
import schedule
import sys
import re
from bs4 import BeautifulSoup
from abc import ABC, abstractmethod
from typing import List, Dict, Any, Set
from google.colab import drive
import shutil

# Selenium setup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# For robust requests with retries
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def get_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    # Use a more recent User-Agent and try to hide automation
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
    options.add_argument('--disable-blink-features=AutomationControlled')

    # Standard robust install for Colab + Linux
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
        return driver
    except Exception as e:
        print(f"Driver Init Error: {e}")
        return None

class JsonStorage:
    def __init__(self, filepath: str = "history.json"):
        self.filepath = filepath
        # Change to Dict[item_id, Dict[str, str]] storing {'date': date_str, 'title': title_str}
        self.seen_items: Dict[str, Dict[str, str]] = self._load()

    def _load(self) -> Dict[str, Dict[str, str]]:
        if not os.path.exists(self.filepath):
            return {}
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # Handle old format (list of strings or dict with only dates) for backward compatibility
                if isinstance(data, list):
                    # Convert old list to new dict format
                    return {item: {'date': '', 'title': ''} for item in data}
                elif isinstance(data, dict):
                    converted_data = {}
                    for item_id, value in data.items():
                        if isinstance(value, str): # Old format: {'item_id': 'date_str'}
                            converted_data[item_id] = {'date': value, 'title': ''}
                        elif isinstance(value, dict) and 'date' in value: # Already in new or similar format
                            converted_data[item_id] = {'date': value.get('date', ''), 'title': value.get('title', '')}
                        else: # Fallback for unexpected formats
                            converted_data[item_id] = {'date': '', 'title': ''}
                    return converted_data
                return {}
        except (json.JSONDecodeError, IOError):
            return {}

    def save(self):
        try:
            with open(self.filepath, 'w', encoding='utf-8') as f:
                json.dump(self.seen_items, f, ensure_ascii=False, indent=2)
        except IOError as e:
            print(f"Error saving history: {e}")

    # is_new now takes item_id, current_date_str, AND current_title_str from the scraped item
    def is_new(self, item_id: str, current_date_str: str, current_title_str: str) -> bool:
        if item_id not in self.seen_items: # Completely new item_id
            return True

        # If item_id is seen, check if the date or title has changed
        stored_item_data = self.seen_items[item_id]
        stored_date = stored_item_data.get('date', '')
        stored_title = stored_item_data.get('title', '')

        if stored_date != current_date_str or stored_title != current_title_str:
            return True

        return False # Item_id seen with the same date and title

    # add now takes item_id, current_date_str, AND current_title_str
    def add(self, item_id: str, current_date_str: str, current_title_str: str):
        self.seen_items[item_id] = {'date': current_date_str, 'title': current_title_str}

class ConsoleNotifier:
    def notify(self, source: str, items: List[Dict[str, Any]]): # Modified to accept a list of items
        for item in items:
            title = item.get('title', 'No Title')
            link = item.get('link', 'No Link')
            date = item.get('date', 'Unknown Date')

            message = (
                f"\n[NEW ALERT] Source: {source}\n"
                f"Title: {title}\n"
                f"Date: {date}\n"
                f"Link: {link}\n"
                f"{'-'*40}"
            )
            print(message)

class BaseScraper(ABC):
    def __init__(self, name: str):
        self.name = name
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36', # Updated User-Agent
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7', # More typical Accept header
            'Accept-Language': 'en-US,en;q=0.9,ko;q=0.8',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1', # Indicate preference for HTTPS
        }
        self.session = self._create_retrying_session()

    def _create_retrying_session(self):
        session = requests.Session()
        retry_strategy = Retry(
            total=5, # Increased retries
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["HEAD", "GET", "OPTIONS"],
            raise_on_status=False
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session

    def _make_request(self, url: str, timeout: int = 20): # Increased timeout
        try:
            response = self.session.get(url, headers=self.headers, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            print(f"[{self.name}] Request failed for {url}: {e}")
            return None

    @abstractmethod
    def get_new_items() -> List[Dict[str, Any]]:
        pass

In [ ]:
class FsbScraper(BaseScraper):
    def __init__(self):
        super().__init__("FSB")
        self.url = "https://www.fsb.org/publications/?policy_area%5B%5D=15&policy_area%5B%5D=31"

    def get_new_items(self) -> List[Dict[str, Any]]:
        items = [] # Initialize items here
        unique_items_data: Dict[str, Dict[str, str]] = {}

        try:
            response = self._make_request(self.url)
            if response is None: return items
            soup = BeautifulSoup(response.content, 'html.parser')

            links = soup.find_all('a')
            for a in links:
                href = a.get('href')
                text = a.get_text(strip=True)
                if not href or len(text) < 5: continue

                full_link = href
                if not href.startswith('http'):
                    full_link = f"https://www.fsb.org{href}"

                # Only process links that match the publication pattern
                if not re.search(r'/\d{4}/\d{2}/', full_link):
                    continue

                date_from_url = ''
                match_url_date = re.search(r'/(\d{4})/(\d{2})/', full_link)
                if match_url_date:
                    y, m = match_url_date.groups()
                    date_from_url = f"{y}-{m}-01"

                date_from_text = ''
                # Try to find a more specific date in text or parent
                parent_to_search = a.find_parent()
                for _ in range(5): # Extend search up to 5 parents
                    if not parent_to_search: break # Corrected: 'parent' to 'parent_to_search'
                    block_text = parent_to_search.get_text(" ", strip=True)
                    match = re.search(r'(\d{1,2})\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+(\d{4})', block_text, re.IGNORECASE)
                    if match:
                         try:
                             d_str = f"{match.group(1)} {match.group(2)} {match.group(3)}"
                             try: parsed = datetime.datetime.strptime(d_str, "%d %B %Y")
                             except ValueError:
                                 try: parsed = datetime.datetime.strptime(d_str, "%d %b %Y")
                                 except ValueError:
                                     parsed = None

                             if parsed:
                                 date_from_text = parsed.strftime("%Y-%m-%d")
                                 break
                         except: pass
                    parent_to_search = parent_to_search.parent

                # Determine the best date for this particular <a> tag
                current_item_date = date_from_text if date_from_text else date_from_url

                # Compare with existing entry for this link, if any
                if full_link not in unique_items_data:
                    unique_items_data[full_link] = {'title': text, 'date': current_item_date}
                else:
                    existing_date = unique_items_data[full_link]['date']

                    # Prioritize: date_from_text (YYYY-MM-DD) > date_from_url (YYYY-MM-01) > empty
                    if not existing_date and current_item_date: # Existing has no date, current has one
                        unique_items_data[full_link]['date'] = current_item_date
                    elif current_item_date: # Both have dates, or existing has date and current has a better one
                        if date_from_text and not existing_date.endswith('-01'): # current is YYYY-MM-DD and existing is not YYYY-MM-01
                             if current_item_date > existing_date: # Both YYYY-MM-DD, pick later
                                 unique_items_data[full_link]['date'] = current_item_date
                        elif date_from_text and existing_date.endswith('-01'): # current is YYYY-MM-DD, existing is YYYY-MM-01. Prioritize current.
                            unique_items_data[full_link]['date'] = current_item_date
                        elif not date_from_text and existing_date.endswith('-01') and current_item_date > existing_date: # Both YYYY-MM-01, pick later
                            unique_items_data[full_link]['date'] = current_item_date
                        elif not date_from_text and not existing_date.endswith('-01') and current_item_date > existing_date: # both YYYY-MM-DD, pick later
                            unique_items_data[full_link]['date'] = current_item_date

                    # Always prefer longer title
                    if len(text) > len(unique_items_data[full_link]['title']): # and not text.lower().startswith('read more'): # added to avoid generic titles
                         unique_items_data[full_link]['title'] = text

            # Convert the unique_items_data dictionary back into a list of items
            for link, data in unique_items_data.items():
                items.append({'id': link, 'title': data['title'], 'link': link, 'date': data['date']})

        except Exception as e:
            print(f"Error scraping FSB: {e}")
        return items

class BcbsScraper(BaseScraper):
    def __init__(self):
        super().__init__("BCBS")
        self.url = "https://www.bis.org/bcbs/publications.htm?m=75&topics=Climate-related+financial+risks&topics=Disclosure"

    def get_new_items(self) -> List[Dict[str, Any]]:
        items = []
        driver = None
        try:
            driver = get_driver()
            if not driver: return items
            driver.get(self.url)
            try:
                WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "tr")))
                time.sleep(3)
            except Exception:
                print("[BCBS] Timeout waiting for table rows.")

            soup = BeautifulSoup(driver.page_source, 'html.parser')
            rows = soup.find_all('tr')
            for row in rows:
                cells = row.find_all('td')
                if not cells: continue

                date_str = '' # Changed default to empty string
                if len(cells) > 0:
                    text = cells[0].get_text(strip=True)
                    if len(text) >= 9 and len(text) <= 20:
                        try:
                            parsed = datetime.datetime.strptime(text, "%d %b %Y")
                            date_str = parsed.strftime("%Y-%m-%d")
                        except: pass

                a_tag = row.find('a')
                if not a_tag: continue
                href = a_tag.get('href')
                if not href: continue

                if '/publ/' in href or '/bcbs/' in href:
                    title = a_tag.get_text(strip=True)
                    if len(title) > 10:
                        full_link = href
                        if not href.startswith('http'):full_link = f"https://www.bis.org{href}"
                        items.append({'id': full_link, 'title': title, 'link': full_link, 'date': date_str})
        except Exception as e:
            print(f"Error scraping BCBS (Selenium): {e}")
        finally:
            if driver: driver.quit()
        return items

class NgfsScraper(BaseScraper):
    def __init__(self):
        super().__init__("NGFS")
        self.url = "https://www.ngfs.net/en/press-release"

    def get_new_items(self) -> List[Dict[str, Any]]:
        items = []
        try:
            response = self._make_request(self.url)
            if response is None: return items
            soup = BeautifulSoup(response.content, 'html.parser')

            links = soup.find_all('a', href=True)
            seen_links = set()

            for a in links:
                href = a.get('href')
                text = a.get_text(strip=True)

                if not href or not text: continue
                if "/en/press-release/" not in href: continue
                if "?page=" in href: continue

                if href in seen_links: continue
                seen_links.add(href)

                full_link = href
                if not href.startswith('http'):
                    full_link = f"https://www.ngfs.net{href}"

                date_str = '' # Changed default to empty string

                # Attempt to find a date within the 'Published on' or 'Updated on' format
                parent = a.find_parent()
                for _ in range(5): # Check parents for date string
                    if not parent: break
                    block_text = parent.get_text(" ", strip=True)
                    match = re.search(r'(\d{1,2})(?:st|nd|rd|th)? of (Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* (\d{4})', block_text, re.IGNORECASE)
                    if match:
                        try:
                            day = match.group(1)
                            month_name = match.group(2)
                            year = match.group(3)
                            # Handle ordinal suffixes (1st, 2nd, 3rd, 4th etc.) - already extracted by regex
                            d_str = f"{day} {month_name} {year}"
                            parsed_date = datetime.datetime.strptime(d_str, "%d %b %Y")
                            date_str = parsed_date.strftime("%Y-%m-%d")
                            break
                        except ValueError:
                            pass
                    parent = parent.parent

                # Fallback: If date is still default, try finding a year in the text
                if date_str == '': # Check against empty string now
                    match_year = re.search(r'(202[3-9])', text)
                    if match_year:
                         date_str = f"{match_year.group(1)}-01-01"

                items.append({'id': full_link, 'title': text, 'link': full_link, 'date': date_str})

        except Exception as e:
            print(f"Error scraping NGFS: {e}")
        return items

class PcafScraper(BaseScraper):
    def __init__(self):
        super().__init__("PCAF")
        self.url = "https://carbonaccountingfinancials.com/en/resources#resources"

    def get_new_items(self) -> List[Dict[str, Any]]:
        items = []
        try:
            response = self._make_request(self.url)
            if response is None: return items
            soup = BeautifulSoup(response.content, 'html.parser')
            links = soup.find_all('a')
            for i, a in enumerate(links):
                title = a.get_text(strip=True)
                link = a.get('href')

                if not link or not title:
                    continue

                filter_passed = False
                # Original filtering logic:
                if "financed emissions" in title.lower() or \
                   "facilitated emissions" in title.lower() or \
                   "standard" in title.lower() or \
                   "/en/video/" in link: # New logic for video links
                    filter_passed = True

                if filter_passed:
                    full_link = link
                    if link.startswith('/'):
                        full_link = f"https://carbonaccountingfinancials.com{link}"

                    date_str = ''
                    match = re.search(r'\((\d{4})\)', title)
                    if match:
                        date_str = f"{match.group(1)}-01-01"
                    else:
                        # Try to extract year from the video URL itself
                        video_date_match = re.search(r'/video/(\d{4})-\w+', link)
                        if video_date_match:
                            date_str = f"{video_date_match.group(1)}-01-01"

                    item_to_add = {'id': full_link, 'title': title, 'link': full_link, 'date': date_str}
                    items.append(item_to_add)

        except Exception as e:
            print(f"Error scraping PCAF: {e}")
        return items

class KosifScraper(BaseScraper):
    def __init__(self):
        super().__init__("PCAF Korea (Kosif)")
        self.url = "https://kosif.org/bbs/board.php?bo_table=s2_9_2"

    def get_new_items(self) -> List[Dict[str, Any]]:
        items = []
        driver = None
        try:
            driver = get_driver()
            if not driver: return items
            driver.get(self.url)
            # Use a long sleep after getting the page to allow for full loading and bypass potential bot detection
            time.sleep(15) # Increased sleep time

            soup = BeautifulSoup(driver.page_source, 'html.parser')
            rows = soup.find_all('tr')
            if rows:
                for row in rows:
                    link_tag = row.find('a', href=True)
                    if not link_tag: continue
                    href = link_tag.get('href')
                    if 'wr_id=' not in href: continue

                    date_str = '' # Changed default to empty string
                    cells = row.find_all('td')
                    for cell in cells:
                        text = cell.get_text(strip=True)
                        if (len(text) == 10 or len(text) == 8) and ('-' in text or '.' in text):
                            sep = '-' if '-' in text else '.'
                            parts = text.split(sep)
                            if len(parts) == 3 and all(p.isdigit() for p in parts):
                                year = parts[0]
                                if len(year) == 2: year = '20' + year
                                month = parts[1].zfill(2)
                                day = parts[2].zfill(2)
                                date_str = f"{year}-{month}-{day}"
                                break

                    title = link_tag.get_text(strip=True)
                    items.append({'id': href, 'title': title, 'link': href, 'date': date_str})
        except Exception as e:
            print(f"Error scraping Kosif: {e}")
        finally:
            if driver: driver.quit()
        return items

class FscScraper(BaseScraper):
    def __init__(self):
        super().__init__("금융위원회")
        self.url = "https://www.fsc.go.kr/no010101"

    def get_new_items(self) -> List[Dict[str, Any]]:
        items = []
        try:
            response = self._make_request(self.url)
            if response is None:
                print(f"[FSC Debug] Request for {self.url} failed.")
                return items
            print(f"[FSC Debug] Request for {self.url} successful (status: {response.status_code}).")
            soup = BeautifulSoup(response.content, 'html.parser')

            links = soup.find_all('a', href=True)
            print(f"[FSC Debug] Found {len(links)} <a> tags.")
            seen_links = set()

            for a in links:
                href = a.get('href')
                if not href or '/no010101/' not in href: continue

                path_part = href.split('?')[0]
                if not path_part.split('/')[-1].isdigit():
                    continue

                if href in seen_links: continue
                seen_links.add(href)

                title = a.get_text(strip=True)
                if not title:
                    print(f"[FSC Debug] Skipping link due to empty title: {href}")
                    continue

                full_link = href
                if not href.startswith('http'): full_link = f"https://www.fsc.go.kr{href}"

                date_str = ''
                row_container = a.find_parent('tr')

                if row_container:
                    row_text = row_container.get_text(" ", strip=True)
                    print(f"[FSC Debug] Processing link: {full_link}, Title: '{title}', Row Text: '{row_text[:100]}...'") # Truncate row_text

                    match_yyyy_mm_dd = re.search(r'(\d{4}[-.]\d{2}[-.]\d{2})', row_text)
                    if match_yyyy_mm_dd:
                        try:
                            parsed_date = datetime.datetime.strptime(match_yyyy_mm_dd.group(1).replace('.', '-'), "%Y-%m-%d")
                            date_str = parsed_date.strftime("%Y-%m-%d")
                            print(f"[FSC Debug] Extracted YYYY-MM-DD date: {date_str}")
                        except ValueError:
                            print(f"[FSC Debug] Failed to parse YYYY-MM-DD date from: {match_yyyy_mm_dd.group(1)}")
                            pass

                    if not date_str:
                        match_yy_mm_dd = re.search(r'(\d{2}[-.]\d{2}[-.]\d{2})', row_text)
                        if match_yy_mm_dd:
                            try:
                                full_date_str = f"20{match_yy_mm_dd.group(1).replace('.', '-')}"
                                parsed_date = datetime.datetime.strptime(full_date_str, "%Y-%m-%d")
                                date_str = parsed_date.strftime("%Y-%m-%d")
                                print(f"[FSC Debug] Extracted YY-MM-DD date: {date_str}")
                            except ValueError:
                                print(f"[FSC Debug] Failed to parse YY-MM-DD date from: {match_yy_mm_dd.group(1)}")
                                pass
                    if not date_str:
                        print("[FSC Debug] No date extracted.")

                items.append({'id': full_link, 'title': title, 'link': full_link, 'date': date_str})

        except Exception as e:
            print(f"Error scraping FSC: {e}")
        return items

class FssScraper(BaseScraper):
    def __init__(self):
        super().__init__("금융감독원")
        self.url = "https://www.fss.or.kr/fss/bbs/B0000188/list.do?menuNo=200218"

    def get_new_items(self) -> List[Dict[str, Any]]:
        items = []
        try:
            response = self._make_request(self.url)
            if response is None: return items
            soup = BeautifulSoup(response.content, 'html.parser')
            rows = soup.find_all('tr')
            if rows:
                for row in rows:
                    link_tag = row.find('a', href=True)
                    if not link_tag: continue
                    href = link_tag.get('href')
                    if not href or 'view.do' not in href or 'nttId=' not in href: continue
                    if 'B0000188' not in href: continue
                    title = link_tag.get_text(strip=True)
                    full_link = href
                    if not href.startswith('http'): full_link = f"https://www.fss.or.kr{href}"
                    date_str = ''
                    # Try to find date in sibling td tags.
                    # A common pattern is that the date is in a td element near the link,
                    # often one of the last td elements in the row.
                    cells = row.find_all('td')
                    for cell in cells:
                        text = cell.get_text(strip=True)
                        # More flexible regex to find YYYY-MM-DD or YYYY.MM.DD
                        match_yyyy_mm_dd = re.search(r'(\d{4}[-.]\d{2}[-.]\d{2})', text)
                        if match_yyyy_mm_dd:
                            date_str = match_yyyy_mm_dd.group(1).replace('.', '-')
                            break
                    items.append({'id': full_link, 'title': title, 'link': full_link, 'date': date_str})
        except Exception as e:
            print(f"Error scraping FSS: {e}")
        return items

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from google.colab import userdata

class EmailNotifier(ConsoleNotifier):
    def __init__(self, sender_email: str, sender_password: str, recipient_emails: List[str]):
        super().__init__()
        self.sender_email = sender_email
        self.sender_password = sender_password
        self.recipient_emails = recipient_emails
        self.smtp_server = "smtp.gmail.com" # Assuming Gmail, change if using another provider
        self.smtp_port = 587

    # Modified signature to accept filtered_out_items
    def notify(self, source: str, new_items: List[Dict[str, Any]], filtered_out_items: List[Dict[str, Any]] = None):
        if filtered_out_items is None:
            filtered_out_items = []

        # Console notification (still one per item for clarity in console)
        # Only notify for new_items in console, as filtered_out_items are not considered primary new alerts
        if new_items:
            for item in new_items:
                super().notify(item.get('source', source), [item])

        msg = MIMEMultipart()
        msg['From'] = self.sender_email

        num_new_items = len(new_items)
        num_filtered_out_items = len(filtered_out_items)

        # Determine subject based on presence of new items or filtered out items
        if not new_items: # If no actual NEW ESG/AI items
            subject = "[알림] 오늘자, 새로운 ESG 및 AI 관련 소식이 없습니다."
        else: # If there are new ESG/AI items
            subject_parts = []
            if num_new_items > 0: # This will be true if we are in this 'else' block
                subject_parts.append(f"{num_new_items} 건 탐색")
            if num_filtered_out_items > 0:
                subject_parts.append(f"{num_filtered_out_items} 건 참고자료")
            subject = f"[알림] {source}: {', '.join(subject_parts)}"


        # HTML 시작 (글꼴 스타일 추가 및 인라인 스타일 적용)
        html_body_content = """
        <html>
        <head>
            <style>
                body { font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', '맑은 고딕', '맑은 고딕(제목)', '맑은 고딕(본문)','KoPub돋움체 Medium', 'Apple Color Emoji', 'Segoe UI', 'Helvetica Neue', Arial, sans-serif; font-size: 14px; color: #333; line-height: 1.6; }
                ul { list-style-type: none; padding: 0; margin: 0; }
                li { margin-bottom: 8px; }
                a { color: #0056b3; text-decoration: none; }
                a:hover { text-decoration: underline; }
                .date { font-size: 12px; color: #777; margin-left: 10px; }
                .footer { margin-top: 30px; font-size: 12px; color: #888; border-top: 1px solid #eee; padding-top: 15px; text-align: center; }
                .main-section { margin-bottom: 25px; padding: 15px; border: 2px solid #555; border-radius: 8px; background-color: #f9f9f9; }
                .main-section-title { font-size: 16px; font-weight: bold; color: #333; margin-bottom: 10px; border-bottom: 1px solid #eee; padding-bottom: 5px; }
                .reference-section { margin-top: 25px; border-top: 1px dashed #ccc; padding-top: 20px; }
                .reference-section-title { font-size: 15px; font-weight: bold; color: #555; margin-bottom: 10px; }
            </style>
        </head>
        <body>
        """

        if new_items:
            html_body_content += """
            <div style="font-size: 18px; font-weight: bold; margin-bottom: 20px; color: #000;"> ESG 관련 주요기관 최신 보도자료:</div>
            """
            # Group new_items by their source for the main section
            grouped_new_items: Dict[str, List[Dict[str, Any]]] = {}
            for item in new_items:
                item_source = item.get('source', 'Unknown Source')
                if item_source not in grouped_new_items:
                    grouped_new_items[item_source] = []
                grouped_new_items[item_source].append(item)

            for item_source, source_items in grouped_new_items.items():
                # Sort items within each source by date, newest first
                # Items without a date (empty string) will be placed at the end.
                source_items.sort(key=lambda x: x.get('date', ''), reverse=True)

                # Apply inline styles for source-group and source-title
                html_body_content += f"""
                <div class="main-section">
                    <div class="main-section-title">{item_source}</div>
                    <ul>
                """
                for item in source_items:
                    title = item.get('title', 'No Title')
                    link = item.get('link', 'No Link')
                    date = item.get('date', 'Unknown Date')

                    html_body_content += f"""
                        <li> <span style="font-weight: bold;"><a href="{link}" style="color: #0056b3; text-decoration: none;">{title}</a></span> / <span style="font-size: 12px; color: #777;">{date}</span></li>
                    """
                html_body_content += """
                    </ul>
                </div>
                """
        else: # if not new_items
            html_body_content += """
            <div style="font-size: 16px; font-weight: bold; margin-bottom: 20px;">오늘자 주요기관 보도자료에서 새로운 ESG 및 AI 관련 소식이 없습니다.</div>
            """


        html_body_content += """
            <div class="footer">
                <p>--</p>
                <p>주요기관: 금융위원회, 금융감독원, PCAF Korea(Kosif), BCBS, PCAF, NGFS, FSB.</p>
            </div>
        </body>
        </html>
        """

        # Add the section for filtered_out_items
        if filtered_out_items:
            html_body_content += """
            <div class="reference-section">
                <div class="reference-section-title" style="font-size: 12px; font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', '맑은 고딕', '맑은 고딕(제목)', '맑은 고딕(본문)','KoPub돋움체 Medium', 'Apple Color Emoji', 'Segoe UI', 'Helvetica Neue', Arial, sans-serif;">(참고) ESG & AI 외 보도자료 (금융위원회 / 금융감독원)</div>
                <ul>
            """
            # Sort filtered out items by date, newest first for better readability
            filtered_out_items.sort(key=lambda x: x.get('date', ''), reverse=True)
            for item in filtered_out_items:
                title = item.get('title', 'No Title')
                link = item.get('link', 'No Link')
                html_body_content += f"""
                    <li style="font-size: 12px; font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', '맑은 고딕', '맑은 고딕(제목)', '맑은 고딕(본문)','KoPub돋움체 Medium', 'Apple Color Emoji', 'Segoe UI', 'Helvetica Neue', Arial, sans-serif;"> <a href="{link}" style="color: #0056b3; text-decoration: none;">{title}</a></li>
                """
            html_body_content += """
                </ul>
            </div>
            """

        msg['Subject'] = subject
        msg.attach(MIMEText(html_body_content, 'html')) # Changed to 'html' subtype

        try:
            server = smtplib.SMTP(self.smtp_server, self.smtp_port)
            server.starttls() # Secure the connection
            server.login(self.sender_email, self.sender_password)
            text = msg.as_string()
            for recipient_email in self.recipient_emails:
                msg['To'] = recipient_email # Set 'To' header for each recipient
                if new_items or filtered_out_items:
                    print(f"[EmailNotifier] Email sent for {source} (New: {len(new_items)}, Filtered Out: {len(filtered_out_items)}) to {recipient_email}")
                else:
                    print(f"[EmailNotifier] Email sent (no new items or filtered out items) to {recipient_email}")
                server.sendmail(self.sender_email, recipient_email, text)
            server.quit()
        except Exception as e:
            print(f"[EmailNotifier] Failed to send email for {source}: {e}")

In [ ]:
def job():
    print(f"[{datetime.datetime.now()}] Starting crawler job...")
    storage = JsonStorage(filepath=file_path)
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    from google.colab import userdata # Import userdata here
    import re # Ensure re module is imported here for re.sub

    # Retrieve email credentials from Colab secrets
    sender_email = userdata.get('SENDER_EMAIL')
    sender_password = userdata.get('SENDER_PASSWORD') # App-specific password for Gmail

    # Initialize notifiers
    notifiers = [
        ConsoleNotifier(),
    ]
    email_notifier = None
    if sender_email and sender_password and recipient_emails:
        email_notifier = EmailNotifier(sender_email, sender_password, recipient_emails)
        print("[EmailNotifier] Email notifications are ENABLED.")
    else:
        print("[EmailNotifier] Email credentials not fully configured. Skipping email notifications.")

    scrapers: list[BaseScraper] = [
        FscScraper(),
        FssScraper(),
        KosifScraper(),
        BcbsScraper(),
        PcafScraper(),
        NgfsScraper(),
        FsbScraper()
    ]

    all_new_items_consolidated = []
    all_filtered_out_items_consolidated = [] # New list to collect filtered items for the '참고' section

    # Define keywords for filtering FSS/FSC items끄
    keywords_for_fss_fsc = ["ESG", "기후", "전환", "녹색", "택소노미", "Taxonomy", "NDC", "탄소", "AI", "인공지능", "모형", "리스크", "Risk", "RMF", "MRM", "빅데이터", "데이터", "ML", "모델", "머신러닝", "DL", "딥러닝", "심사", "대출"]

    for scraper in scrapers:
        try:
            items = scraper.get_new_items()
            print(f"[{scraper.name}] Found {len(items)} items.")

            items_to_email_for_current_scraper = []

            for item in items:
                if not isinstance(item, dict):
                    print(f"[DEBUG] Expected dict, but found type: {type(item)} for item: {item}")
                    continue

                item_id = item.get('id') or item.get('link')
                item_date = item.get('date', '')
                original_item_title = item.get('title', '')

                # DEBUG: Print original title before cleaning
                # print(f"DEBUG: Original title before cleaning: '{original_item_title}'")

                # Post-processing: remove specific phrase from title using regex for robustness
                cleaned_title = re.sub(r'금일\s*등록된\s*게시글', '', original_item_title).strip()
                item['title'] = cleaned_title # Update the item dictionary with the cleaned title

                # DEBUG: Print cleaned title in the item dictionary
                # print(f"DEBUG: Item title in dictionary after cleaning: '{item['title']}'")

                if not item_id:
                    print(f"[DEBUG] Item missing 'id' or 'link': {item}")
                    continue

                if storage.is_new(item_id, item_date, cleaned_title): # Use cleaned_title for history check
                    # All newly detected items are notified to console.
                    # They will be processed further for email based on filters.
                    item['source'] = scraper.name
                    for notifier in notifiers: # ConsoleNotifier
                        notifier.notify(scraper.name, [item])

                    # Apply keyword filtering specifically for FSS and FSC for email purposes
                    if scraper.name in ["금융위원회", "금융감독원"]:
                        matched_keyword = False
                        for keyword in keywords_for_fss_fsc:
                            if keyword.lower() in cleaned_title.lower(): # Use cleaned_title for keyword matching
                                matched_keyword = True
                                break

                        if matched_keyword:
                            items_to_email_for_current_scraper.append(item)
                            storage.add(item_id, item_date, cleaned_title) # Store item if it's going to be emailed
                        else:
                            all_filtered_out_items_consolidated.append(item) # Add to the list for the '참고' section
                            storage.add(item_id, item_date, cleaned_title) # Store item even if filtered out, to avoid re-notification
                    else: # For other scrapers, no keyword filtering for email
                        items_to_email_for_current_scraper.append(item)
                        storage.add(item_id, item_date, cleaned_title) # Store item

            # Add items that passed filters to the consolidated list for the main email section
            all_new_items_consolidated.extend(items_to_email_for_current_scraper)

        except Exception as e:
            print(f"Error checking {scraper.name}: {e}")

    # Send a single consolidated email after all scrapers have run
    if email_notifier:
        email_notifier.notify('주요기관 보도자료', all_new_items_consolidated, all_filtered_out_items_consolidated) # Updated call

    # Always save history after all scrapers have run
    storage.save()

    print("Job finished.")

In [ ]:
job()

In [ ]:
0

In [ ]:
# 테스트 코드 Load the history.json file
with open(file_path, 'r', encoding='utf-8') as f:
    loaded_data = json.load(f)

items_to_remove = [
    {
        'id': 'https://www.fsc.go.kr/no010101/86732?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '정부, 전국 요양시설 종신보험 가입실태 전수조사 실시 및 관리 강화.'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86725?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '중동 상황에도 흔들림 없이 생산적 금융으로의 대전환을 추진합니다. - 이억원 금융위원장, 제5차 생산적 금융 대전환 회의 개최'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86722?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': 'URL 링크가 포함된 고유가 피해지원금 문자와 알림은 100% 사기입니다'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86721?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '중복상장 제도개선 공개세미나 개최'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86719?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '사업보고서 등에 대한 조사·감리결과 조치 - 제7차 금융위원회(’26.4.15.) 조치 의결'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86717?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '국민성장펀드, 네이버에 AI 데이터센터, GPU 도입자금 4,000억원 저리대출 승인'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86712?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '생성형 AI 활용을 위한 혁신금융서비스 지정 절차가 간소화되었습니다. - 제7차 금융위원회 정례회의(‘26.4.15.) 보고'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86710?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '「자본시장특별사법경찰 집무규칙」 개정안 금융위원회(4.15일) 의결'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86709?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '아직 미흡한 동네 병·의원에서의 스마트폰 실손보험금 청구(‘실손24’)도 빨리 확대되도록, 다양한 개선책을 적극 추진하겠습니다.'
    },
    {
        'id': 'https://www.fsc.go.kr/no010101/86701?srchCtgry=&curPage=&srchKey=&srchText=&srchBeginDt=&srchEndDt=',
        'date': '',
        'title': '해수부·금융위, 호르무즈 해협 통항 대비 우리 선사의 애로사항 해소에 힘 합쳐 -선사 보험료 부담 경감 및 신속 가입 여건 마련을 위해 원팀 대응-'
    }
]

print("Attempting to remove specified items from history.json...")
for item_info in items_to_remove:
    item_id = item_info['id']
    expected_date = item_info['date']
    expected_title = item_info['title']

    if item_id in loaded_data:
        stored_item = loaded_data[item_id]
        stored_date = stored_item.get('date', '')
        stored_title = stored_item.get('title', '')

        if stored_date == expected_date and stored_title == expected_title:
            del loaded_data[item_id]
            print(f"Successfully removed entry: {item_id} (Date: {expected_date}, Title: {expected_title})")
        else:
            print(f"Found entry for {item_id}, but date/title did not match. Stored: {stored_item}")
    else:
        print(f"Entry for {item_id} not found in history.json.")

# Save the updated data back to history.json
with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(loaded_data, f, ensure_ascii=False, indent=2)
print("\nUpdated history.json saved.")

# Print the updated loaded_data to confirm changes
print("\nUpdated loaded_data:")
print(loaded_data)
